In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import os
from glob import glob

from skl2onnx.helpers.onnx_helper import load_onnx_model
from tqdm import tqdm

from service.fragment.net import Net
from utilities.dataloader_generator import generate_dataloader
from utilities.evaluate_model import eval_original_model_onnx
from utilities.get_accuracy import accuracy_score_net
from utilities.get_data_score import get_data_score
from utilities.get_macs import get_macs_params_onnx
from utilities.get_net_score import get_score_net
from utilities.load_dataset_chest_xray import load_dataset
from utilities.visual import draw_stitchNet_fromTuples


2025-11-24 12:08:17.085052621 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [3]:
data_score = get_data_score(16)

batch_size = 16
dataset_train = load_dataset()
dataset_val = load_dataset("test")
dataloaders = dict(
    train=generate_dataloader(dataset_train, batch_size=batch_size),
    val=generate_dataloader(dataset_val, batch_size=batch_size),
)
os.makedirs("../_results_chest_xray/original", exist_ok=True)


In [4]:
# Evaluate accuracy of the original networks (fine-tuned for chest X-ray)
modelnames = sorted(glob("../_results_chest_xray/finetune/*/model_ft.onnx"))
for i, modelname in tqdm(enumerate(modelnames), position=0, leave=True):
    namewithoutext = modelname.split("/")[-2]
    if namewithoutext == "densenet121":
        continue

    model_onnx1 = load_onnx_model(modelname)
    macs, params = get_macs_params_onnx(model_onnx1)
    valacc, trainacc = eval_original_model_onnx(model_onnx1, dataloaders)

    # Get score
    fragmentFiles = sorted(glob(f"../_results_chest_xray/fragments/net{i:03}/*.onnx"))
    onnxFragments = []
    js = []
    for j, fragmentFile in enumerate(fragmentFiles):
        onnxFragment = load_onnx_model(fragmentFile)
        onnxFragments.append(onnxFragment)
        js.append((i, j))
    net1 = Net(onnxFragments, i)
    score = get_score_net(net1, data_score)

    accuracy = accuracy_score_net(Net([model_onnx1]), dataset_val, bs=256)
    print("ACC:", accuracy, valacc, trainacc)

    with open(f"../_results_chest_xray/original/{namewithoutext}.txt", "w") as f:
        f.write(f'{valacc},{trainacc},{macs},{params},{score},"{tuple(js)}"\n')

    draw_stitchNet_fromTuples(
        js, name=f"../_results_chest_xray/original/{namewithoutext}"
    )


0it [00:00, ?it/s]

Node Init Time Elapsed 0.00026702880859375
Tensor Init Time Elapsed 0.1717360019683838
IO Tensor Init Time Elapsed 1.8596649169921875e-05
Constant Search Time Elapsed 1.1920928955078125e-05
Update Nodes Tensors  Time Elapsed 4.553794860839844e-05
{'Conv': 2.09808349609375e-05, 'Relu': 1.9073486328125e-05, 'MaxPool': 1.33514404296875e-05, 'AveragePool': 3.0994415283203125e-06, 'Reshape': 1.2159347534179688e-05, 'Gemm': 8.821487426757812e-06}


/media/shafigh/Disk/new-stitchnet/.venv/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
100%|██████████| 150/150 [00:09<00:00, 15.21it/s]
7it [00:06,  1.11it/s]
100%|██████████| 2391/2391 [00:19<00:00, 121.12it/s]
1it [01:11, 71.89s/it]/media/shafigh/Disk/new-stitchnet/.venv/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


ACC: 0.959012965286491 0.959012965286491 0.9706040124450166
Node Init Time Elapsed 0.011298418045043945
Tensor Init Time Elapsed 0.002732992172241211
IO Tensor Init Time Elapsed 7.939338684082031e-05
Constant Search Time Elapsed 3.9577484130859375e-05
Update Nodes Tensors  Time Elapsed 0.0005066394805908203
{'Conv': 0.0001590251922607422, 'HardSwish': 4.1484832763671875e-05, 'Relu': 3.0994415283203125e-05, 'ReduceMean': 4.172325134277344e-05, 'HardSigmoid': 1.8358230590820312e-05, 'Mul': 3.838539123535156e-05, 'Add': 2.5033950805664062e-05, 'Reshape': 7.152557373046875e-06, 'Gemm': 7.62939453125e-06}


100%|██████████| 150/150 [00:02<00:00, 50.79it/s]
12it [00:00, 20.82it/s]
100%|██████████| 2391/2391 [00:04<00:00, 573.78it/s]
3it [01:31, 25.95s/it]

ACC: 0.7749895441237976 0.7749895441237976 0.7719128848835962
Node Init Time Elapsed 0.011710166931152344
Tensor Init Time Elapsed 0.03194284439086914
IO Tensor Init Time Elapsed 9.012222290039062e-05
Constant Search Time Elapsed 3.7670135498046875e-05
Update Nodes Tensors  Time Elapsed 0.0004904270172119141
{'Conv': 0.0001461505889892578, 'Relu': 9.751319885253906e-05, 'MaxPool': 7.867813110351562e-06, 'Add': 6.031990051269531e-05, 'ReduceMean': 1.049041748046875e-05, 'Reshape': 7.3909759521484375e-06, 'Gemm': 3.5762786865234375e-06}


/media/shafigh/Disk/new-stitchnet/.venv/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
100%|██████████| 150/150 [00:32<00:00,  4.60it/s]
5it [00:02,  1.97it/s]
100%|██████████| 2391/2391 [00:41<00:00, 58.06it/s]
4it [05:25, 97.72s/it]

ACC: 0.7327478042659975 0.7327478042659975 0.7288917498122519
Node Init Time Elapsed 0.0004372596740722656
Tensor Init Time Elapsed 0.37856268882751465
IO Tensor Init Time Elapsed 5.030632019042969e-05
Constant Search Time Elapsed 1.5735626220703125e-05
Update Nodes Tensors  Time Elapsed 9.751319885253906e-05
{'Conv': 4.291534423828125e-05, 'Relu': 3.361701965332031e-05, 'MaxPool': 1.8358230590820312e-05, 'AveragePool': 3.814697265625e-06, 'Reshape': 1.1920928955078125e-05, 'Gemm': 7.867813110351562e-06}


/media/shafigh/Disk/new-stitchnet/.venv/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
100%|██████████| 150/150 [02:03<00:00,  1.22it/s]
15it [00:21,  1.46s/it]
100%|██████████| 2391/2391 [02:21<00:00, 16.88it/s]
5it [17:36, 211.37s/it]

ACC: 0.9694688414889168 0.9694688414889168 0.9788649286557236
